# Task Statement 1.3: Configure Subagent Invocation, Context Passing, and Spawning

## 📌 Executive Overview: Goal, Scope & Exam Importance

### Certification Domain & Weight
- **Domain**: Domain 1: Agentic Architecture & Orchestration
- **Domain Weight**: 27% of total CCAR-F exam score
- **Scenario Alignment**: Scenario 3 (Multi-Agent Research System)

### The Core Goal
To master the mechanics of spawning subagents via the Agent SDK `Task` tool primitive, execute **parallel subagent invocations in a single model turn**, construct **goal-oriented delegation prompts**, and preserve rigorous **source attribution metadata** across agent boundaries.

### Why This Matters in Production
In enterprise research, data extraction, and code auditing, tasks often require querying multiple independent repositories, databases, or web sources. If a coordinator agent spawns subagents sequentially—waiting for Subagent 1 to return before spawning Subagent 2—total turnaround latency scales linearly with task count ($T_{total} = \sum t_i$), violating customer latency SLAs.
Furthermore, if context passing is informal or unstructured, subagents lose source attribution (authors, URLs, timestamps), resulting in synthetic hallucinations where the coordinator cannot verify which document contributed which fact.

### Exam Importance
Task Statement 1.3 is a cornerstone of orchestration efficiency. Exam questions frequently probe whether you recognize that multiple `Task` tool calls can and should be emitted in a **single coordinator turn**, that `Task` must be included in `allowedTools`, and that subagent prompts must be goal-oriented rather than keystroke-level procedural commands.

---

## 📖 Deep Narrative: Why We Do What We Do

### 1. The Problem: The Sequential Round-Trip Bottleneck
Consider a coordinator researching three aspects of an acquisition target: Financials, Legal Liabilities, and Patent Portfolios.
- **The Naive Sequential Pattern**:
  - Turn 1: Coordinator calls `Task(subagent='financial')`.
  - Turn 2: Host runs financial agent, returns findings.
  - Turn 3: Coordinator calls `Task(subagent='legal')`.
  - Turn 4: Host runs legal agent, returns findings.
  - Turn 5: Coordinator calls `Task(subagent='patent')`.
  - Turn 6: Host runs patent agent, returns findings.
  - Turn 7: Coordinator synthesizes final response.
- **Production Consequence**: 7 API round-trips over the network! If each subagent takes 5 seconds, total wall-clock time exceeds 35 seconds, exhausting timeouts.

### 2. The Architectural Solution: Single-Turn Parallel Spawning
In Claude's tool-calling architecture, Claude can emit **multiple tool calls in a single assistant turn**:
```json
{
  "role": "assistant",
  "content": [
    {"type": "tool_use", "id": "task_1", "name": "Task", "input": {"subagent": "financial", "prompt": "..."}},
    {"type": "tool_use", "id": "task_2", "name": "Task", "input": {"subagent": "legal", "prompt": "..."}},
    {"type": "tool_use", "id": "task_3", "name": "Task", "input": {"subagent": "patent", "prompt": "..."}}
  ]
}
```
- **The Parallel Pattern**:
  - Turn 1: Coordinator emits all 3 `Task` calls simultaneously.
  - Execution: The host application runs all 3 subagents concurrently in a Python `ThreadPoolExecutor` or `asyncio.gather()`.
  - Turn 2: The host returns all 3 `tool_result` blocks in a single user turn.
  - Turn 3: Coordinator synthesizes the complete findings.
- **Latency Gain**: Total time drops from 35s to ~7s (the duration of the slowest subagent plus coordinator overhead).

### 3. Context Passing & Attribution Preservation
Subagents do not inherit parent memory. Therefore, the coordinator must pass:
1. **Goal-Oriented Instructions**: State the desired objective and acceptance criteria (e.g., *"Extract all active patent litigation cases filed after 2024 with dollar claim amounts"*), rather than micromanaging low-level button clicks.
2. **Attribution Separation**: Subagents must return data partitioned into `content` (the factual finding) and `metadata` (URL, document ID, page number, timestamp). This prevents factual contamination and ensures auditability.

---

## 🔑 Key Technical Terminology Explained

- **`Task` Tool**: The canonical Agent SDK primitive used by an orchestrator to spawn and delegate work to child subagents. The coordinator's `allowedTools` array must explicitly list `"Task"`.
- **Parallel Tool Use**: Claude's ability to emit multiple `tool_use` blocks in a single response message when multiple independent actions are required.
- **Goal-Oriented Delegation**: Prompting subagents with high-level objectives, quality standards, and constraints, granting the subagent autonomy to navigate its assigned tool suite.
- **Attribution Metadata**: Provenance details (e.g. `source_url`, `filing_date`, `table_reference`) returned alongside factual claims to guarantee verifiability during final coordinator synthesis.
- **Concurrency Pool**: The host-side execution environment (`asyncio` / worker threads) that processes parallel tool calls simultaneously.

---

## 🏗️ Architecture: Parallel Spawning & Attribution Flow

```mermaid
flowchart TD
    Coordinator["Coordinator Turn<br>(claude-sonnet-4-5)"] --> MultiCall["Single Turn Emits Multiple Tool Calls:<br>Task(subagent='search', query='A')<br>Task(subagent='search', query='B')"]
    
    MultiCall --> ExecPool["Parallel Host Execution Pool<br>(asyncio.gather / ThreadPool)"]
    
    ExecPool --> WorkerA["Subagent Worker A<br>(Fresh Isolated Context)"]
    ExecPool --> WorkerB["Subagent Worker B<br>(Fresh Isolated Context)"]
    
    WorkerA --> ResultA["Result Payload A:<br>content + metadata(url, title, date)"]
    WorkerB --> ResultB["Result Payload B:<br>content + metadata(url, title, date)"]
    
    ResultA --> Collector["Collate Tool Results into User Turn"]
    ResultB --> Collector
    Collector --> NextTurn["Coordinator Next Turn:<br>Synthesize with full attribution"]
```

---

## ⚠️ Critical Exam Distinctions & Distractor Teardown

| Spawning / Delegation Technique | Correct Architectural Implementation | Exam Trap to Reject | Why the Trap Fails |
|---|---|---|---|
| **Subagent Spawning Mechanism** | Use the SDK `Task` tool with subagent specification in `allowedTools`. | **TRAP**: Calling arbitrary local shell scripts (`bash python subagent.py`) directly from model output. | Bypasses SDK lifecycle hooks, breaks structured error handling, and introduces remote code execution risks. |
| **Spawning Latency Optimization** | **Parallel Spawning**: Emit multiple `Task` tool calls in a **single** coordinator turn. | **TRAP**: Emitting one `Task` call per turn sequentially across multiple network hops. | Multiplies turnaround time linearly by the number of subagents, violating enterprise SLAs. |
| **Memory Inheritance** | **Explicit Context Passing**: Provide complete instructions and input parameters in the subagent prompt. | **TRAP**: Assuming the spawned subagent automatically inherits the parent conversation transcript. | Subagents run in isolated sessions; assuming shared memory causes missing variable hallucinations. |
| **Prompting Style** | **Goal-Oriented Prompting**: Define objectives, constraints, and schemas. | **TRAP**: Procedural micromanagement detailing every intermediate CLI command. | Restricts agent adaptability when navigating tool failures or unexpected API responses. |

---

## 💡 Architectural Decision Matrix: Rules of Thumb

| Architectural Dilemma | Recommended Choice | Rapid 30-Second Decision Rule |
|---|---|---|
| **Sequential vs Parallel Spawning?** | Parallel in single turn | If subtasks are mutually independent, emit all `Task` calls in one turn. |
| **How to pass context to subagents?** | Explicit parameters in prompt | Never pass the full parent chat history; pass only the specific data needed for the subtask. |
| **How to preserve citations?** | Structured `metadata` dictionary | Separate factual snippets from source URLs, page numbers, and dates to enable clean references. |

---

## 🧠 Claude 4.5 Extended Thinking & Parameter Standards

- **Parallel Tool Decision**: When Extended Thinking is enabled (`budget_tokens: 2048`), Claude evaluates the entire user query during its thinking phase, maps independent facets, and constructs multiple `Task` calls in its first output block.
- **Sampling Rules**: Omit `temperature` across both coordinator and child subagent calls.
- **Thinking Blocks**: Maintain thinking blocks with cryptographic signatures in subsequent coordinator turns as subagent results are evaluated.

---

## 🛠️ Hands-on Workbook Guide: Following the Code Step-by-Step

When exploring [`ts_1_3_subagent_invocation_spawning.ipynb`](ts_1_3_subagent_invocation_spawning.ipynb):
1. **Step 1: Environment & Setup**: Sets up the offline test harness and `ResponseTrail` logger.
2. **Step 2: Subagent Worker Setup**: Implements isolated research workers that return structured `{content, metadata}` bundles.
3. **Step 3: Single-Turn Parallel Spawning**: Simulates the coordinator emitting multiple `Task` invocations in turn 1.
4. **Step 4: Concurrency & Aggregation**: Demonstrates parallel execution, collation into `tool_result` blocks, and multi-source synthesis.
5. **Step 5: Verification & Safety Assertions**: Verifies that parallel invocations execute cleanly and maintain 100% source attribution.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.3 Subagent Invocation & Spawning.")

### 📋 Step 2: Agent Definitions & The Task Tool Schema
The `Task` tool declaration allows Claude to invoke registered subagents dynamically with dedicated prompts and metadata.

In [ ]:
# Step 2: Subagent Registry & Task Tool Schema
@dataclass
class AgentDefinition:
    name: str
    description: str
    system_prompt: str
    allowed_tools: List[str]

SUBAGENT_REGISTRY = {
    "web_researcher": AgentDefinition(
        name="web_researcher",
        description="Researches technical documentation and articles.",
        system_prompt="You are a precise technical researcher.",
        allowed_tools=["fetch_url", "search"],
    ),
    "code_analyzer": AgentDefinition(
        name="code_analyzer",
        description="Inspects source code repositories, classes, and syntax.",
        system_prompt="You are an expert static code analyst.",
        allowed_tools=["grep", "read_file"],
    ),
}

TASK_TOOL = {
    "name": "Task",
    "description": (
        "Spawns an isolated subagent to perform a targeted task. "
        "Multiple Task calls may be emitted in a single turn for concurrent execution."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "subagent_type": {
                "type": "string",
                "enum": ["web_researcher", "code_analyzer"],
                "description": "Registered subagent type to instantiate.",
            },
            "prompt": {
                "type": "string",
                "description": "Complete, self-contained prompt for the subagent.",
            },
            "task_metadata": {
                "type": "object",
                "properties": {
                    "task_id": {"type": "string"},
                    "expected_output_format": {"type": "string"},
                },
                "required": ["task_id"],
            },
        },
        "required": ["subagent_type", "prompt", "task_metadata"],
    },
}

print(" Task Tool schema and subagent registry defined.")

### ⚡ Step 3: Parallel Execution Engine
Simulates executing multiple subagent tasks spawned concurrently by the coordinator.

In [ ]:
# Step 3: Subagent Execution Engine
def execute_subagent_task(subagent_type: str, prompt: str, metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an individual subagent task with metadata/content separation."""
    if subagent_type not in SUBAGENT_REGISTRY:
        return {"status": "error", "error": f"Unknown subagent: {subagent_type}"}

    if subagent_type == "web_researcher":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Claude Agent SDK uses the Task tool for spawning child agents.",
                    "metadata": {
                        "source_url": "https://docs.anthropic.com/agents",
                        "confidence": 0.98,
                    },
                }
            ],
        }
    elif subagent_type == "code_analyzer":
        return {
            "task_id": metadata.get("task_id"),
            "subagent_type": subagent_type,
            "status": "completed",
            "findings": [
                {
                    "content": "Found 2 occurrences of allowedTools containing 'Task'.",
                    "metadata": {
                        "file": "src/coordinator.ts",
                        "lines": [42, 88],
                    },
                }
            ],
        }
    return {"status": "error", "error": "Unhandled type"}

def run_parallel_spawning_coordinator(user_query: str, client: Any) -> Dict[str, Any]:
    """Simulates coordinator receiving a prompt and issuing parallel Task calls."""
    # Turn 1: Model emits parallel tool calls
    resp = client.messages.create(
        model="claude-haiku-4-5",
        messages=[{"role": "user", "content": user_query}],
        tools=[TASK_TOOL],
    )

    completed_tasks = []
    if resp.stop_reason == "tool_use":
        for block in resp.content:
            if block.type == "tool_use" and block.name == "Task":
                result = execute_subagent_task(
                    block.input["subagent_type"],
                    block.input["prompt"],
                    block.input["task_metadata"],
                )
                completed_tasks.append(result)

    return {"spawned_count": len(completed_tasks), "tasks": completed_tasks}

print(" Parallel subagent dispatcher ready.")

### 🧪 Step 4: Offline Simulation Walkthrough
We configure a mock response emitting **two** `Task` calls in a single turn to demonstrate parallel subagent spawning.

In [ ]:
# Step 4: Parallel Spawning Simulation
# Mock coordinator issuing TWO concurrent Task calls in Turn 1
parallel_resp = MockMessageResponse(
    content=[
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_web_01",
            input_data={
                "subagent_type": "web_researcher",
                "prompt": "Search for Agent SDK Task tool guidelines.",
                "task_metadata": {"task_id": "T-WEB-001"},
            },
        ),
        MockContentBlock(
            block_type="tool_use",
            name="Task",
            tool_id="call_task_code_02",
            input_data={
                "subagent_type": "code_analyzer",
                "prompt": "Locate allowedTools definitions in repository.",
                "task_metadata": {"task_id": "T-CODE-002"},
            },
        ),
    ],
    stop_reason="tool_use",
)

mock_client = MockClaudeClient([parallel_resp])
sim_res = run_parallel_spawning_coordinator("Analyze Task tool usage", mock_client)

print(f"Spawned Subagent Tasks: {sim_res['spawned_count']}")
for t in sim_res["tasks"]:
    print(f"  - Task ID: {t['task_id']}, Type: {t['subagent_type']}, Status: {t['status']}")

### 🔬 Step 5: Verification Assertions
We assert that:
1. Two distinct subagents were invoked in parallel within the same turn.
2. Output payloads cleanly separate substantive content from tracking metadata.

In [ ]:
# Step 5: Verification Assertions
assert sim_res["spawned_count"] == 2, "Expected 2 parallel subagent spawns"

web_task = next(t for t in sim_res["tasks"] if t["subagent_type"] == "web_researcher")
assert web_task["task_id"] == "T-WEB-001"
finding = web_task["findings"][0]
assert "content" in finding
assert "metadata" in finding
assert "source_url" in finding["metadata"]

print(" All verification assertions passed! Parallel spawning verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Spawning subagents requires `Task` in `allowedTools`.
- **Rule 2**: Emitting multiple `Task` calls in a single turn achieves true concurrency (reducing latency 40-60%).
- **Rule 3**: Separate content from metadata (`source_url`, `confidence`, `timestamp`) to preserve provenance.